In [1]:
from delta.tables import DeltaTable
from pyspark.sql.functions import (
    col, current_timestamp, datediff, expr, lit, when
)
from pyspark.sql.types import DateType, DecimalType
from pyspark.sql import Window
import pyspark.sql.functions as F
from datetime import date

SRC_TABLE       = "raw_accounts"
TARGET_TABLE    = "silver_accounts"
WATERMARK_TABLE = "_pipeline_watermarks"
LAYER_KEY       = "bronze_accounts"    # shares watermark with Bronze accounts

print("Config loaded.")

StatementMeta(, 25d31800-7608-45df-8782-1d4613d1eb64, 3, Finished, Available, Finished, False)

Config loaded.


In [2]:
last_watermark = (
    spark.table(WATERMARK_TABLE).filter(f"layer_name = '{LAYER_KEY}'")
         .select("watermark_ts").collect()[0]["watermark_ts"]
)

df_raw = spark.table(SRC_TABLE)

# ingestion_timestamp only exists on rows added by nb_bronze_accounts_incremental.
# The original Day 2 rows (from the Copy Activity) don't have it.
# If the column is missing, process everything — this is effectively the first run.
if "ingestion_timestamp" in df_raw.columns:
    df_raw_new = df_raw.filter(col("ingestion_timestamp") > last_watermark)
else:
    print("ingestion_timestamp column not found — processing all rows as first run.")
    df_raw_new = df_raw

print(f"New raw_accounts rows to process: {df_raw_new.count()}")

if df_raw_new.count() == 0:
    spark.stop()
    notebookutils.notebook.exit("NO_NEW_DATA")

StatementMeta(, 25d31800-7608-45df-8782-1d4613d1eb64, 4, Finished, Available, Finished, False)

New raw_accounts rows to process: 500


In [3]:
if "updated_at" in df_raw_new.columns:
    # Incremental rows from nb_bronze_accounts_incremental — order by updated_at
    window_dedup = Window.partitionBy("account_id").orderBy(col("updated_at").desc())
    df_latest = (
        df_raw_new
        .withColumn("rn", F.row_number().over(window_dedup))
        .filter(col("rn") == 1)
        .drop("rn")
    )
else:
    # Original Day 2 data — no updated_at, just deduplicate by account_id
    print("updated_at not found — deduplicating by account_id only (first run).")
    df_latest = df_raw_new.dropDuplicates(["account_id"])

# Safely drop metadata columns whether they exist or not
for c in ["ingestion_timestamp", "updated_at"]:
    if c in df_latest.columns:
        df_latest = df_latest.drop(c)

print(f"Unique accounts after dedup: {df_latest.count()}")

StatementMeta(, 25d31800-7608-45df-8782-1d4613d1eb64, 5, Finished, Available, Finished, False)

Unique accounts after dedup: 500


In [4]:
def mask_email(c): return expr(f"concat(substring({c},1,1),'***@',substring({c},instr({c},'@')+1,length({c})))")
def mask_phone(c): return expr(f"concat(repeat('X',length({c})-4),substring({c},length({c})-3,4))")
def mask_pan(c):   return expr(f"concat('XXXXX',substring({c},6,5))")

def age_bucket(dob):
    today = date.today().isoformat()
    return (
        when(datediff(lit(today), col(dob)) / 365.25 < 26, "18-25")
        .when(datediff(lit(today), col(dob)) / 365.25 < 36, "26-35")
        .when(datediff(lit(today), col(dob)) / 365.25 < 51, "36-50")
        .otherwise("51+")
    )

df_silver = (
    df_latest
    .withColumn("email_masked",      mask_email("email"))
    .withColumn("phone_masked",      mask_phone("phone"))
    .withColumn("pan_masked",        mask_pan("pan"))
    .withColumn("age_bucket",        age_bucket("date_of_birth"))
    .withColumn("account_open_date", col("account_open_date").cast(DateType()))
    .withColumn("account_balance",   col("account_balance").cast(DecimalType(15, 2)))
    .withColumn("silver_updated_at", current_timestamp())
    .drop("email", "phone", "pan", "date_of_birth", "updated_at")
    .select(
        "account_id", "customer_name", "email_masked", "phone_masked",
        "pan_masked", "age_bucket", "home_city", "branch",
        "account_open_date", "account_balance", "risk_category",
        "kyc_status", "account_status", "silver_updated_at"
    )
)
print(f"Silver rows ready to MERGE: {df_silver.count()}")

StatementMeta(, 25d31800-7608-45df-8782-1d4613d1eb64, 6, Finished, Available, Finished, False)

Silver rows ready to MERGE: 500


In [5]:
if spark.catalog.tableExists(TARGET_TABLE):
    dt = DeltaTable.forName(spark, TARGET_TABLE)
    (dt.alias("t").merge(df_silver.alias("s"), "t.account_id = s.account_id")
       .whenMatchedUpdateAll()
       .whenNotMatchedInsertAll()
       .execute())
    print(f"MERGE complete → {TARGET_TABLE}")
else:
    df_silver.write.mode("overwrite").format("delta").saveAsTable(TARGET_TABLE)
    print(f"First run — wrote {TARGET_TABLE} from scratch")

spark.stop()
print("Silver accounts incremental complete.")

StatementMeta(, 25d31800-7608-45df-8782-1d4613d1eb64, 7, Finished, Available, Finished, False)

MERGE complete → silver_accounts
Silver accounts incremental complete.
